In [1]:
import csv, json, math, time
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

def find_root():
    """Walk up from the notebook's folder until we find the folder containing dataset_infos."""
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "dataset_infos").exists():
            return q
    raise FileNotFoundError("Could not find the Radar_ML_lite folder")

ROOT = find_root()
DATA_DIR = ROOT / "dataset"
SPLIT_CSV = ROOT / "dataset_infos" / "split_manifest.csv"

FS = 120e6                                   # sample rate: 120 million samples/second
CLASSES = ["barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad"]
SNRS = [30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30]
SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

print("root:", ROOT)
print("device:", device)
print("npy files:", len(list(DATA_DIR.glob("*_recorded.npy"))))

root: C:\Users\Think pro\Desktop\Radar_ML_Lite_DATATHON\Radar_ML_Lite_supervised_multiclass_classification\Radar_ML_lite
device: cuda
npy files: 660


In [2]:
from collections import Counter

def get_label(filename):
    token = filename.split("_")[1].split("#")[0].lower()
    return "barker" if token.startswith("barker") else token

with open(SPLIT_CSV, newline="") as f:
    rows = list(csv.DictReader(f))

train_rows = [r for r in rows if r["split"] == "train"]
val_rows   = [r for r in rows if r["split"] == "validation"]

print("train:", len(train_rows), "validation:", len(val_rows))
print("train pulses per class:", dict(Counter(get_label(r["file"]) for r in train_rows)))
print("val pulses per class:  ", dict(Counter(get_label(r["file"]) for r in val_rows)))

train: 52800 validation: 13200
train pulses per class: {'barker': 5760, 'cw': 960, 'fmcw': 7680, 'fsk': 15360, 'hyp': 7680, 'nlfm': 7680, 'quad': 7680}
val pulses per class:   {'barker': 1440, 'cw': 240, 'fmcw': 1920, 'fsk': 3840, 'hyp': 1920, 'nlfm': 1920, 'quad': 1920}
